In [1]:
#!/usr/bin/env python3
"""
Ray Tune Hyperparameter Search with Feature Selection and K-Fold CV
"""

import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib
import random
import pickle
import numpy as np
import pandas as pd
import tempfile
import argparse
from datetime import datetime

from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
import xgboost as xgb

import ray
from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler

import config
from preprocessing_utils import ClinicalPreprocessorWrapper, MrnaPreprocessorWrapper, MutationPreprocessorWrapper
from model_utils import *

    
# Set seeds for reproducibility
set_random_seed(seed=config.SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

def train_with_raytune(config_params):
    """
    Ray Tune trainable function for hyperparameter optimization with k-fold CV.
    Includes optional feature selection before training.
    """
    # Import inside function for Ray workers
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import numpy as np
    import pandas as pd
    import random
    import ray
    from ray import tune
    from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
    from sklearn.model_selection import KFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.feature_selection import SelectFromModel
    import config
    from model_utils import GeneSelector, ModalityEncoder, build_optimizer
    from preprocessing_utils import ClinicalPreprocessorWrapper, MrnaPreprocessorWrapper, MutationPreprocessorWrapper
    import xgboost as xgb

    # Define MultimodalNet inside trainable
    class MultimodalNet(nn.Module):
        def __init__(self,
                     clin_dim, mrna_dim, mut_dim,
                     clin_hidden=config.CLIN_HIDDEN,
                     mrna_hidden=config.MRNA_HIDDEN,
                     mut_hidden=config.MUT_HIDDEN,
                     clin_dropout=config.CLIN_DROPOUT,
                     mrna_dropout=config.MRNA_DROPOUT,
                     mut_dropout=config.MUT_DROPOUT,
                     activation=config.ACTIVATION_FUNC,
                     fusion_hidden=config.FUSION_HIDDEN,
                     fusion_dropout=config.FUSION_DROPOUT,
                     use_gene_sel=True,
                    ):
            super().__init__()

            self.use_gene_sel = use_gene_sel
            if use_gene_sel:
                self.gene_sel_mrna = GeneSelector(mrna_dim)
                self.gene_sel_mut = GeneSelector(mut_dim)

            self.enc_clin = ModalityEncoder(clin_dim, clin_hidden, clin_dropout, activation)
            self.enc_mrna = ModalityEncoder(mrna_dim, mrna_hidden, mrna_dropout, activation)
            self.enc_mut  = ModalityEncoder(mut_dim, mut_hidden, mut_dropout, activation)

            total_dim = clin_hidden[-1] + mrna_hidden[-1] + mut_hidden[-1]
            self.fusion_fc = nn.Sequential(
                nn.Linear(total_dim, fusion_hidden),
                nn.ReLU(),
                nn.Dropout(fusion_dropout),
                nn.Linear(fusion_hidden, 1)
            )

        def forward(self, clin, mrna, mut):
            if self.use_gene_sel:
                mrna = self.gene_sel_mrna(mrna)
                mut = self.gene_sel_mut(mut)

            clin_emb = self.enc_clin(clin)
            mrna_emb = self.enc_mrna(mrna)
            mut_emb  = self.enc_mut(mut)

            fused = torch.cat([clin_emb, mrna_emb, mut_emb], dim=1)
            output = self.fusion_fc(fused)
            return output.squeeze()

    def to_loader(c, m, mu, y, shuffle=False):
        def check_numeric(df, name):
            if isinstance(df, np.ndarray):
                df = pd.DataFrame(df)
            non_numeric_cols = []
            for col in df.columns:
                if not pd.api.types.is_numeric_dtype(df[col]):
                    non_numeric_cols.append(col)
            if non_numeric_cols:
                print(f"WARNING: {name} has non-numeric columns: {non_numeric_cols}")
            return df.to_numpy(dtype=np.float32)

        c = check_numeric(c, "Clinical")
        m = check_numeric(m, "mRNA")
        mu = check_numeric(mu, "Mutation")

        if isinstance(y, (pd.DataFrame, pd.Series)):
            y = y.to_numpy(dtype=np.float32).reshape(-1, 1)
        else:
            y = np.array(y, dtype=np.float32).reshape(-1, 1)
        y = y.squeeze()

        ds = TensorDataset(
            torch.tensor(c),
            torch.tensor(m),
            torch.tensor(mu),
            torch.tensor(y)
        )
        return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=shuffle)

    # Retrieve data from Ray's object store
    clin_all = ray.get(config_params["clinical_train_ref"])
    mrna_all = ray.get(config_params["mrna_train_ref"])
    mut_all = ray.get(config_params["mutation_train_ref"])
    y_all = ray.get(config_params["y_train_ref"])

    # Set seed
    seed = config_params["seed"]
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Get k-fold parameters
    n_folds = config_params["n_folds"]
    fs_method = config_params.get("fs_method", "NoFeatureSelection")
    
    # K-Fold Cross-Validation
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    
    # Store metrics across folds
    fold_aurocs = []
    fold_auprcs = []
    fold_f1s = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(clin_all)):
        # Split data for this fold
        clin_train = clin_all.iloc[train_idx] if hasattr(clin_all, 'iloc') else clin_all[train_idx]
        clin_val = clin_all.iloc[val_idx] if hasattr(clin_all, 'iloc') else clin_all[val_idx]
        
        mrna_train = mrna_all.iloc[train_idx] if hasattr(mrna_all, 'iloc') else mrna_all[train_idx]
        mrna_val = mrna_all.iloc[val_idx] if hasattr(mrna_all, 'iloc') else mrna_all[val_idx]
        
        mut_train = mut_all.iloc[train_idx] if hasattr(mut_all, 'iloc') else mut_all[train_idx]
        mut_val = mut_all.iloc[val_idx] if hasattr(mut_all, 'iloc') else mut_all[val_idx]
        
        y_train = y_all.iloc[train_idx] if hasattr(y_all, 'iloc') else y_all[train_idx]
        y_val = y_all.iloc[val_idx] if hasattr(y_all, 'iloc') else y_all[val_idx]

        # ===== PREPROCESSING (fit on train fold, transform both) =====
        clinical_prep = ClinicalPreprocessorWrapper(
            cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
            categorical_cols=config.CATEGORICAL_COLS,
            max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
            uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
        )
        mrna_prep = MrnaPreprocessorWrapper(
            max_null_frac=config.MAX_NULL_FRAC,
            uniform_thresh=config.UNIFORM_THRESHOLD,
            random_state=seed,
        )
        mutation_prep = MutationPreprocessorWrapper(
            max_mutation_count=config_params.get("max_mutation_count", config.MAX_MUTATION_COUNT),
            uniform_thresh=config_params.get("mutation_uniform_thresh", config.MUTATION_UNIFORM_THRESH),
        )

        clinical_prep.fit(clin_train)
        mrna_prep.fit(mrna_train, y_train)
        mutation_prep.fit(mut_train)

        clin_train = clinical_prep.transform(clin_train)
        clin_val = clinical_prep.transform(clin_val)

        mrna_train = mrna_prep.transform(mrna_train)
        mrna_val = mrna_prep.transform(mrna_val)

        mut_train = mutation_prep.transform(mut_train)
        mut_val = mutation_prep.transform(mut_val)

        # ===== FEATURE SELECTION (fit on train fold) =====
        def apply_feature_selection(estimator, mrna_train, mrna_val, mut_train, mut_val, y_train):
            """Helper function to apply SelectFromModel to both mRNA and mutation data"""
            threshold = config_params.get("fs_threshold", "median")
            max_features = config_params.get("fs_max_features", None)

            # Create and fit SelectFromModel for mRNA
            sfm_mrna = SelectFromModel(estimator, threshold=threshold, max_features=max_features)
            sfm_mrna.fit(mrna_train, y_train)

            # Create and fit SelectFromModel for mutations
            sfm_mut = SelectFromModel(estimator, threshold=threshold, max_features=max_features)
            sfm_mut.fit(mut_train, y_train)

            # Transform data
            mrna_train_fs = sfm_mrna.transform(mrna_train)
            mrna_val_fs = sfm_mrna.transform(mrna_val)
            mut_train_fs = sfm_mut.transform(mut_train)
            mut_val_fs = sfm_mut.transform(mut_val)

            return mrna_train_fs, mrna_val_fs, mut_train_fs, mut_val_fs

        # Create estimator based on feature selection method
        if fs_method == "LogisticRegression":
            fs_estimator = LogisticRegression(
                C=config_params.get("fs_C", 1.0),
                penalty=config_params.get("fs_penalty", "l2"),
                max_iter=1000,
                solver='saga',
                random_state=seed
            )
            mrna_train, mrna_val, mut_train, mut_val = apply_feature_selection(
                fs_estimator, mrna_train, mrna_val, mut_train, mut_val, y_train
            )

        elif fs_method == "RandomForest":
            fs_estimator = RandomForestClassifier(
                n_estimators=config_params.get("fs_n_estimators", 100),
                max_depth=config_params.get("fs_max_depth", None),
                random_state=seed
            )
            mrna_train, mrna_val, mut_train, mut_val = apply_feature_selection(
                fs_estimator, mrna_train, mrna_val, mut_train, mut_val, y_train
            )

        elif fs_method == "XGBoost":
            if xgb is None:
                raise ImportError("XGBoost not available")
            fs_estimator = xgb.XGBClassifier(
                learning_rate=config_params.get("fs_learning_rate", 0.1),
                max_depth=config_params.get("fs_max_depth", 6),
                n_estimators=config_params.get("fs_n_estimators", 100),
                random_state=seed,
                use_label_encoder=False,
                eval_metric='logloss'
            )
            mrna_train, mrna_val, mut_train, mut_val = apply_feature_selection(
                fs_estimator, mrna_train, mrna_val, mut_train, mut_val, y_train
            )

        # NoFeatureSelection: skip feature selection (no transformation needed)

        # Create data loaders with preprocessed (and possibly feature-selected) data
        train_loader = to_loader(clin_train, mrna_train, mut_train, y_train, shuffle=True)
        val_loader = to_loader(clin_val, mrna_val, mut_val, y_val)

        # Get dimensions after preprocessing/feature selection
        clin_dim = clin_train.shape[1]
        mrna_dim = mrna_train.shape[1]
        mut_dim = mut_train.shape[1]

        # Create model
        model = MultimodalNet(
            clin_dim=clin_dim,
            mrna_dim=mrna_dim,
            mut_dim=mut_dim,
            clin_hidden=config_params["clin_hidden"],
            mrna_hidden=config_params["mrna_hidden"],
            mut_hidden=config_params["mut_hidden"],
            clin_dropout=config_params["clin_dropout"],
            mrna_dropout=config_params["mrna_dropout"],
            mut_dropout=config_params["mut_dropout"],
            activation=config_params.get("activation", "relu"),
            fusion_hidden=config_params["fusion_hidden"],
            fusion_dropout=config_params["fusion_dropout"],
            use_gene_sel=False  # No gene selection in model
        ).to(device)

        # Optimizer and loss
        optimizer = build_optimizer(model, config_params)

        pos_weight_value = torch.tensor(
            (len(y_train) - y_train.sum()) / y_train.sum(),
            dtype=torch.float32
        ).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)

        # Training loop for this fold
        best_val_auroc = 0
        patience_counter = 0
        patience = config_params["patience"]
        num_epochs = config_params["num_epochs"]

        for epoch in range(num_epochs):
            # Training
            model.train()
            for clin, mrna, mut, y in train_loader:
                clin = clin.to(device)
                mrna = mrna.to(device)
                mut = mut.to(device)
                y = y.to(device)

                optimizer.zero_grad()
                outputs = model(clin, mrna, mut)
                loss = criterion(outputs, y)
                loss.backward()
                optimizer.step()

            # Validation
            model.eval()
            all_probs, all_labels = [], []
            with torch.no_grad():
                for clin, mrna, mut, y in val_loader:
                    clin = clin.to(device)
                    mrna = mrna.to(device)
                    mut = mut.to(device)
                    y = y.to(device)

                    probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                    all_probs.append(probs)
                    all_labels.append(y.cpu().numpy())

            all_probs = np.concatenate(all_probs)
            all_labels = np.concatenate(all_labels)

            # Calculate metrics
            val_auroc = roc_auc_score(all_labels, all_probs)
            val_auprc = average_precision_score(all_labels, all_probs)

            # Find optimal threshold for F1
            thresholds = np.linspace(0.001, 0.9, 200)
            best_f1 = 0
            for t in thresholds:
                preds = (all_probs >= t).astype(float)
                f1 = f1_score(all_labels, preds, zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1

            # Early stopping
            if val_auroc > best_val_auroc:
                best_val_auroc = val_auroc
                best_val_auprc = val_auprc
                best_val_f1 = best_f1
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        
        # Store best metrics from this fold
        fold_aurocs.append(best_val_auroc)
        fold_auprcs.append(best_val_auprc)
        fold_f1s.append(best_val_f1)
    
    # Report average metrics across all folds
    tune.report({
        "auroc": np.mean(fold_aurocs),
        "auprc": np.mean(fold_auprcs),
        "f1": np.mean(fold_f1s),
        "epoch": num_epochs  # Report final epoch
    })


def get_search_space(fs_method, clinical_train_ref, mrna_train_ref, mutation_train_ref, y_train_ref, n_folds):
    """
    Returns the search space for Ray Tune based on the feature selection method.
    """
    # Base search space (same for all methods)
    base_space = {
        # Data references (training data only)
        "clinical_train_ref": clinical_train_ref,
        "mrna_train_ref": mrna_train_ref,
        "mutation_train_ref": mutation_train_ref,
        "y_train_ref": y_train_ref,

        # Fixed parameters
        "seed": config.SEED,
        "patience": config.PATIENCE,
        "num_epochs": config.NUM_EPOCHS,
        "n_folds": n_folds,

        # Feature selection method
        "fs_method": fs_method,

        # Neural network architecture hyperparameters
        "clin_hidden": tune.choice([
            [64],
            [128],
            [128, 64],
        ]),
        "mrna_hidden": tune.choice([
            [64],
            [128],
            [128, 64],
            [256, 128],
            [128, 64, 32],
        ]),
        "mut_hidden": tune.choice([
            [64],
            [128],
            [64, 32],
            [128, 64],
            [64, 32, 16],
        ]),
        "clin_dropout": tune.choice([
            [0.0],
            [0.2],
            [0.3],
        ]),
        "mrna_dropout": tune.choice([
            [0.0],
            [0.2],
            [0.3],
        ]),
        "mut_dropout": tune.choice([
            [0.0],
            [0.2],
            [0.3],
        ]),
        "fusion_hidden": tune.choice([64, 128, 256]),
        "fusion_dropout": tune.uniform(0.0, 0.5),
        "lr": tune.loguniform(1e-5, 1e-4),
        "activation": tune.choice(["leaky_relu", "relu", "gelu", "silu"]),

        # Mutation preprocessing hyperparameters
        "max_mutation_count": tune.choice([1, 5, 10, 15]),
        "mutation_uniform_thresh": tune.uniform(0.90, 0.99),

        # Optimizer selection
        "optimizer": tune.choice(["Adam", "AdamW", "SGD", "RMSprop"]),

        # Optimizer-specific hyperparameters
        # weight_decay (for Adam, AdamW, SGD, RMSprop)
        "weight_decay": tune.loguniform(1e-6, 1e-2),

        # SGD-specific
        "momentum": tune.uniform(0.8, 0.99),  # Only used if optimizer=SGD
        "nesterov": tune.choice([True, False]),  # Only used if optimizer=SGD

        # RMSprop-specific
        "rmsprop_alpha": tune.uniform(0.9, 0.999),  # Only used if optimizer=RMSprop
    }

    # Add feature selection specific hyperparameters
    if fs_method == "LogisticRegression":
        base_space.update({
            "fs_C": tune.loguniform(0.001, 10.0),
            "fs_penalty": tune.choice(["l1", "l2"]),
            "fs_threshold": tune.choice(["mean", "median", "0.5*mean"]),
            "fs_max_features": tune.choice([None, 100, 200, 500]),
        })
    elif fs_method == "RandomForest":
        base_space.update({
            "fs_n_estimators": tune.choice([50, 100, 200]),
            "fs_max_depth": tune.choice([None, 5, 10, 20]),
            "fs_threshold": tune.choice(["mean", "median", "0.5*mean"]),
            "fs_max_features": tune.choice([None, 100, 200, 500]),
        })
    elif fs_method == "XGBoost":
        base_space.update({
            "fs_learning_rate": tune.loguniform(0.01, 0.3),
            "fs_max_depth": tune.choice([3, 5, 7, 9]),
            "fs_n_estimators": tune.choice([50, 100, 200]),
            "fs_threshold": tune.choice(["mean", "median", "0.5*mean"]),
            "fs_max_features": tune.choice([None, 100, 200, 500]),
        })
    # NoFeatureSelection: no additional parameters

    return base_space


Using device: cuda


In [7]:
fs_method = "RandomForest"
num_samples = 10
n_folds = 3  # Number of folds for cross-validation


print(f"\n{'='*80}")
print(f"Ray Tune Hyperparameter Search with {n_folds}-Fold Cross-Validation")
print(f"Feature Selection Method: {fs_method}")
print(f"Number of Trials: {num_samples}")
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

# Load data (training only)
print("Loading training data...")
X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))

clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))
mrna_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mrna_cols.joblib"))
mutation_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mutation_cols.joblib"))

# Split by modality
clinical_train = X_train[clinical_cols]
mrna_train = X_train[mrna_cols]
mutation_train = X_train[mutation_cols]

print(f"Data loaded: {len(X_train)} training samples")
print(f"Will perform {n_folds}-fold cross-validation")

# Initialize Ray
if not ray.is_initialized():
    ray.init(ignore_reinit_error=True)

# Put data in Ray's object store
print("Loading data into Ray object store...")
clinical_train_ref = ray.put(clinical_train)
mrna_train_ref = ray.put(mrna_train)
mutation_train_ref = ray.put(mutation_train)
y_train_ref = ray.put(y_train)

# Get search space
search_space = get_search_space(
    fs_method,
    clinical_train_ref, mrna_train_ref, mutation_train_ref, y_train_ref,
    n_folds
)

# Configure scheduler and reporter
scheduler = ASHAScheduler(
    metric="f1",
    mode="max",
    max_t=config.NUM_EPOCHS,
    grace_period=10,
    reduction_factor=2
)

reporter = CLIReporter(
    metric_columns=["auroc", "auprc", "f1", "epoch"],
    max_report_frequency=30
)

# Resources per trial
resources_per_trial = {
    "cpu": 1,
    "gpu": 1
}

print(f"\nStarting Ray Tune search...")
print(f"Resources per trial: {resources_per_trial}")

# Run Ray Tune
result = tune.run(
    train_with_raytune,
    config=search_space,
    resources_per_trial=resources_per_trial,
    num_samples=num_samples,
    scheduler=scheduler,
    progress_reporter=reporter,
    storage_path=os.path.expanduser("~/UCEC_Recurrence/ray_results"),   # <── persistent & writable
    name=f"raytune_{fs_method}_kfold",
    verbose=1,
    raise_on_failed_trial=False,
    resume="AUTO"         
)

# Get best trial
best_trial = result.get_best_trial("f1", "max", "last")

print(f"\n{'='*80}")
print(f"Best trial results:")
print(f"  F1: {best_trial.last_result['f1']:.4f}")
print(f"  AUROC: {best_trial.last_result['auroc']:.4f}")
print(f"  AUPRC: {best_trial.last_result['auprc']:.4f}")
print(f"{'='*80}\n")

# Extract hyperparameters (exclude data references and fixed params)
best_hyperparams = {
    k: v for k, v in best_trial.config.items()
    if not k.endswith("_ref") and k not in ["seed", "patience", "num_epochs", "n_folds"]
}

# Save results
output_prefix = f"{fs_method}_raytune_kfold"

# Save best hyperparameters
with open(f'{output_prefix}_best_hyperparams.pkl', 'wb') as f:
    pickle.dump(best_hyperparams, f)
print(f"Saved best hyperparameters to '{output_prefix}_best_hyperparams.pkl'")

# Save experiment results
experiment_results = {
    "fs_method": fs_method,
    "best_hyperparams": best_hyperparams,
    "best_metrics": {
        "f1": best_trial.last_result['f1'],
        "auroc": best_trial.last_result['auroc'],
        "auprc": best_trial.last_result['auprc'],
    },
    "num_samples": num_samples,
    "n_folds": n_folds,
    "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "all_trials_df": result.dataframe(),
}

with open(f'{output_prefix}_results.pkl', 'wb') as f:
    pickle.dump(experiment_results, f)
print(f"Saved experiment results to '{output_prefix}_results.pkl'")

# Save trial dataframe as CSV for easy viewing
result.dataframe().to_csv(f'{output_prefix}_all_trials.csv', index=False)
print(f"Saved all trials to '{output_prefix}_all_trials.csv'")

print(f"\n{'='*80}")
print(f"Experiment completed successfully!")
print(f"Feature Selection: {fs_method}")
print(f"Best F1: {best_trial.last_result['f1']:.4f}")
print(f"End Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

# Shutdown Ray
ray.shutdown()

2025-12-12 12:34:28,917	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2025-12-12 12:34:28,983	ERROR tune_controller.py:235 -- Failed to restore the run state.
Traceback (most recent call last):
  File "/oscar/home/gchermsi/env-with-ray-py/lib/python3.11/site-packages/ray/tune/execution/tune_controller.py", line 230, in __init__
    self.resume(resume_config=resume_config)
  File "/oscar/home/gchermsi/env-with-ray-py/lib/python3.11/site-packages/ray/tune/execution/tune_controller.py", line 437, in resume
    raise ValueError(
ValueError: Tried to resume experiment from directory '/users/gchermsi/UCEC_Recurrence/ray_results/raytune_RandomForest_kfold', but no experiment state file of the form 'experiment_state-{}.json' was found. This is expected if you are launching a new experiment.



Ray Tune Hyperparameter Search with 3-Fold Cross-Validation
Feature Selection Method: RandomForest
Number of Trials: 10
Start Time: 2025-12-12 12:34:28

Loading training data...
Data loaded: 147 training samples
Will perform 3-fold cross-validation
Loading data into Ray object store...

Starting Ray Tune search...
Resources per trial: {'cpu': 1, 'gpu': 1}


2025-12-12 12:34:28,985	INFO tune_controller.py:238 -- Restarting experiment.


== Status ==
Current time: 2025-12-12 12:34:29 (running for 00:00:00.18)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 0/32 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kfold/driver_artifacts
Number of trials: 10/10 (10 PENDING)


== Status ==
Current time: 2025-12-12 12:34:59 (running for 00:00:30.23)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 1.0/32 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kfold/driver_artifacts
Number 

(train_with_raytune pid=1971574) [2025-12-12 12:35:03,112 E 1971574 1996453] core_worker_process.cc:825: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


...

== Status ==
Current time: 2025-12-12 13:19:04 (running for 00:44:35.17)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 1.0/32 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kfold/driver_artifacts
Number of trials: 10/10 (1 PENDING, 1 RUNNING, 8 TERMINATED)


== Status ==
Current time: 2025-12-12 13:19:34 (running for 00:45:05.25)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 1.0/32 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kf

(train_with_raytune pid=3299060) [2025-12-12 13:22:38,505 E 3299060 3327926] core_worker_process.cc:825: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


== Status ==
Current time: 2025-12-12 13:23:04 (running for 00:48:35.65)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 1.0/32 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kfold/driver_artifacts
Number of trials: 10/10 (1 RUNNING, 9 TERMINATED)


== Status ==
Current time: 2025-12-12 13:23:34 (running for 00:49:05.66)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 1.0/32 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kfold/driver_

2025-12-12 13:26:17,811	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/users/gchermsi/UCEC_Recurrence/ray_results/raytune_RandomForest_kfold' in 0.0540s.
2025-12-12 13:26:17,814	INFO tune.py:1041 -- Total run time: 3108.90 seconds (3108.77 seconds for the tuning loop).


== Status ==
Current time: 2025-12-12 13:26:17 (running for 00:51:48.82)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 640.000: None | Iter 320.000: None | Iter 160.000: None | Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None
Logical resource usage: 1.0/32 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2025-12-12_12-29-28_013350_4122921/artifacts/2025-12-12_12-34-28/raytune_RandomForest_kfold/driver_artifacts
Number of trials: 10/10 (10 TERMINATED)



Best trial results:
  F1: 0.6827
  AUROC: 0.7770
  AUPRC: 0.6901

Saved best hyperparameters to 'RandomForest_raytune_kfold_best_hyperparams.pkl'
Saved experiment results to 'RandomForest_raytune_kfold_results.pkl'
Saved all trials to 'RandomForest_raytune_kfold_all_trials.csv'

Experiment completed successfully!
Feature Selection: RandomForest
Best F1: 0.6827
End Time: 2025-12-12 13:26:17

